# SINCON 월요일 대체공휴일 보정

1단계 submission_clean_refined.csv에 고정된 7개 층을 적용합니다. 월요일 층을 끈 주후보도 대조합니다. 실측은 공식 train의 2026-06-30까지만 사용합니다.

기존·중첩 각각 발동 4개 기준일 모두 전체 RMSLE 개선. 서로 다른 검증 날짜는 2026-03-02·2026-05-25 두 개뿐입니다. 로컬 macOS 검증이며 Kaggle·Colab·Linux 실기 기록이 아닙니다.


In [ ]:
from pathlib import Path
import os,sys,json,importlib.util
sys.dont_write_bytecode=True
for key in ('OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS','VECLIB_MAXIMUM_THREADS','NUMEXPR_NUM_THREADS','BLIS_NUM_THREADS'):
    os.environ[key]='1'
ROOT = ''  # 패키지 폴더. 빈 값이면 제한된 탐색을 사용합니다.
BASELINE_CSV = ''  # 1단계 출력. 빈 값이면 input/ 다음 reference/를 사용합니다.
OUTPUT_DIR = ''  # 빈 값이면 Kaggle /kaggle/working, Colab /content, 로컬 별도 폴더.


In [ ]:
MANIFEST='monday_delivery_manifest.json'
KIND='sincon_monday_holiday_preregistered_v1'
def bounded(root,depth,skip=()):
    root=Path(root);found=[]
    if not root.is_dir() or root.is_symlink():return found
    for directory,children,files in os.walk(root,followlinks=False):
        p=Path(directory);level=len(p.relative_to(root).parts)
        children[:]=sorted(n for n in children if n not in skip and level<depth and not (p/n).is_symlink())
        if MANIFEST in files and not (p/MANIFEST).is_symlink():found.append(p)
    return found
if ROOT:
    packages=[Path(ROOT).expanduser().resolve()]
else:
    packages=[Path.cwd(),Path.cwd()/'monday_delivery']
    packages+=bounded('/content',3,('drive',))+bounded('/kaggle/input',4)
matches=[]
for p in dict.fromkeys(packages):
    if (p/MANIFEST).is_file():
        try:
            if json.loads((p/MANIFEST).read_text()).get('package_kind')==KIND:matches.append(p.resolve())
        except (ValueError,OSError):pass
matches=list(dict.fromkeys(matches))
if len(matches)!=1:raise ValueError('월요일 패키지가 하나여야 합니다. ROOT를 지정하세요: '+str(matches))
PACKAGE=matches[0]
spec=importlib.util.spec_from_file_location('sincon_monday_runtime',PACKAGE/'delivery_runtime.py')
runtime=importlib.util.module_from_spec(spec);spec.loader.exec_module(runtime)
print('패키지:',PACKAGE)


In [ ]:
context=runtime.prepare(PACKAGE,BASELINE_CSV,OUTPUT_DIR)
print(json.dumps(context['baseline_verification'],ensure_ascii=False,indent=2))


In [ ]:
output,layers,layer_report=runtime.build_final(context)
print('적용 순서:',context['final'].LAYER_NAMES)
print('행 수:',len(output))


In [ ]:
disabled=runtime.check_monday_disabled(context)
print('월요일을 끈 주후보 재현:',disabled)


In [ ]:
result=runtime.finish(context,output,layers,layer_report)
